In [1]:
import numpy as np
import pandas as pd
import torch
import json
from collections import defaultdict
from sklearn.preprocessing import StandardScaler



In [ ]:
scholar_df = pd.DataFrame({
    "scholar_id": [1, 2, 3, 4],
    "research_interests": [
        "deep learning for flood prediction and hydrological modelling",
        "computer vision and medical image analysis",
        "structural health monitoring using machine learning",
        "remote sensing and climate change"
    ],
    "expertise": [
        "deep learning, hydrology, remote sensing",
        "computer vision, deep learning, medical imaging",
        "machine learning, structural engineering, sensors",
        "remote sensing, GIS, climate modelling"
    ],
    "department": ["Civil Engineering","Computer Science","Civil Engineering","Civil Engineering"],
    "university": ["IIT Kharagpur","IIT Delhi","IIT Bombay","IIT Kharagpur"],
    "country": ["India","India","India","India"],
    "publication_count": [  12, 25, 18, 30],
    "citation_count": [ 120, 450, 200, 600],
    "years_experience": [ 2, 5, 3, 7 ]
})

scholar_df

In [15]:
df = pd.read_json(r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\phd\sop\gpt_extracted_data.json")
df.head(2)

,name,research_interests,expertise,department,university,country,publications,publication_count,citation_count,years_experience,sop_paragraphs,category,sub_domain,scholar_id,source_file,sop_paragraphs_json,full_sop_markdown,word_count
0,Maya L. Patel,"[Autonomous navigation for aerial robotics, Mu...","[Python, C++, Robot Operating System (ROS), Te...",Electrical and Computer Engineering,Stanford University,United States,[Learning Safe Maneuvers for Quadrotor Swarms ...,2,34,3,[From watching a flock of drones weave through...,Engineering & Technology,AI & Autonomous Systems,150,generated_sop_150.pdf,"[""From watching a flock of drones weave throug...",From watching a flock of drones weave through ...,430
1,Alejandro R. Gómez,[Satellite remote sensing for climate change m...,"[Python, R, MATLAB, Google Earth Engine, ENVI,...",Civil and Environmental Engineering,University of Cambridge,United Kingdom,[Deep Learning‑Based Atmospheric Correction fo...,2,21,4,[When I first examined a series of Sentinel‑2 ...,Engineering & Technology,AI & Autonomous Systems,151,generated_sop_151.pdf,"[""When I first examined a series of Sentinel\u...",When I first examined a series of Sentinel‑2 i...,445


In [16]:
df.at[2, "full_sop_markdown"]

'From the moment I programmed my first line‑of‑code to stabilize a hobbyist quadrotor, I have been fascinated by the promise of autonomous systems that can safely operate in complex, dynamic environments. My undergraduate capstone project, which integrated visual odometry with inertial measurements to achieve sub‑centimeter accuracy in indoor flight, revealed both the immense potential and the critical safety challenges that lie at the intersection of AI and robotics.\n\nMy central research questions revolve around (i) how to guarantee safety and reliability in multi‑agent aerial platforms while preserving learning efficiency, and (ii) how to embed explainable decision‑making mechanisms within edge‑deployed AI so that operators can trust autonomous behaviors in real time. To address these questions I propose two core pillars: (a) constrained reinforcement learning that incorporates formal safety specifications, and (b) lightweight sensor‑fusion pipelines optimized for embedded GPUs, en

In [ ]:
text_col = ["research_interests", "publications", "full_sop_markdown", "expertise"]
categorical_col = ["department", "university", "country", "sub_domain", "category"]
numerical_col = ["publication_count", "years_experience", "citation_count"]

In [18]:
prof_df = pd.read_csv(r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prof\extracted_04_prof_data.csv")
prof_df.head(2)

,name,expertise,department,title,topics_display_name,concepts_display_name,primary_topics_display_name,abstract
0,Jaya Madan,Electrical and Electronic Engineering,Department of Electronics and Communication En...,Optimizing Tin-based Solar Cells: Unleashing t...,"{'Physical Sciences', 'Materials Science', 'En...","{'Materials science', 'Organic chemistry', 'Re...","{'Materials science', 'Organic chemistry', 'Re...",A sustainable and environmentally conscious fu...
1,Jaya Madan,Electrical and Electronic Engineering,Department of Electronics and Communication En...,Enhancing Photovoltaic Performance of Lead-Fre...,"{'Physical Sciences', 'Materials Chemistry', '...","{'Engineering physics', 'Optoelectronics', 'Ma...","{'Engineering physics', 'Optoelectronics', 'Ma...",Tin-based photovoltaic (PV) cells have become ...


In [20]:
prof_df.at[2, "title"]

'Enhancing the Performance of CsSnCl<sub>3</sub>-Based Perovskite Solar Cells Through SCAPS Simulation Optimization'

In [33]:
text_col = ["title", "abstract", "topics_display_name", "concepts_display_name", "primary_topics_display_name", "expertise"]
categorical_col = ["department", ]
numerical_col = [""]

In [34]:
df1 = df[["department", "university", "country", "sub_domain", "category"]]
df2 = prof_df[["department"]]
df2.head()

,department
0,Department of Electronics and Communication En...
1,Department of Electronics and Communication En...
2,Department of Electronics and Communication En...
3,Department of Electronics and Communication En...
4,Department of Electronics and Communication En...


In [32]:
df2.at[900,"expertise"]

'Drug Design and Synthesis, Drug Screening'

In [25]:
df2["department"] = df2["department"].str.replace("Department of", "", regex=False)
df2.head()

C:\Users\ps302\AppData\Local\Temp\ipykernel_32040\3212051313.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2["department"] = df2["department"].str.replace("Department of", "", regex=False)


,expertise,department
0,Electrical and Electronic Engineering,Electronics and Communication Engineering
1,Electrical and Electronic Engineering,Electronics and Communication Engineering
2,Electrical and Electronic Engineering,Electronics and Communication Engineering
3,Electrical and Electronic Engineering,Electronics and Communication Engineering
4,Electrical and Electronic Engineering,Electronics and Communication Engineering


In [ ]:
merged_df = pd.cat

In [28]:
# temp = df.loc[df["name"] == "Balasubramanian P"]
# temp.iloc[1]

In [35]:
vocab_dict = defaultdict()
# categorical_columns = ["department", "university", "country"]


In [37]:
# 1. Identify all categorical columns to map
columns_to_map = ["department", "university", "country", "sub_domain", "category", "expertise"]

# 2. Build the unified categorical mapping
categorical_maps = {}

for col in columns_to_map:
    # Gather values from both df1 and df2 (if the column exists in them)
    series_list = []
    if col in df1.columns:
        series_list.append(df1[col])
    if col in df2.columns:
        series_list.append(df2[col])
        
    # Safety Check: If the column isn't in either DataFrame, skip it
    if len(series_list) == 0:
        print(f"Warning: Column '{col}' not found in either df1 or df2. Skipping.")
        continue
    elif len(series_list) == 1:
        combined_values = series_list[0].fillna("UNKNOWN").astype(str)
    else:
        combined_values = pd.concat(series_list).fillna("UNKNOWN").astype(str)
    
    # Get sorted unique values (excluding the 'UNKNOWN' placeholder to avoid duplicate keys)
    unique_values = sorted(list(set(combined_values.unique()) - {"UNKNOWN"}))
    
    # Map sorted values starting from 1, reserving 0 for UNKNOWN
    mapping = {value: idx + 1 for idx, value in enumerate(unique_values)}
    mapping["UNKNOWN"] = 0
    
    categorical_maps[col] = mapping

# 3. Apply the token mappings to df1 and df2 to create ID columns
df1_mapped = df1.copy()
for col in df1.columns:
    if col in categorical_maps:
        mapping = categorical_maps[col]
        df1_mapped[col] = df1[col].fillna("UNKNOWN").astype(str).map(lambda x: mapping.get(x, 0))

df2_mapped = df2.copy()
for col in df2.columns:
    if col in categorical_maps:
        mapping = categorical_maps[col]
        df2_mapped[col] = df2[col].fillna("UNKNOWN").astype(str).map(lambda x: mapping.get(x, 0))

# Display the mapped heads
print("\n--- df1 mapped ---")
print(df1_mapped.head())
print("\n--- df2 mapped ---")
print(df2_mapped.head())



--- df1 mapped ---
   department  university  country  sub_domain  category
0         117           8        7           1         4
1          17          14        6           1         4
2         117           7        7           1         4
3          17          14        6           1         4
4         116           7        7           1         4

--- df2 mapped ---
   department
0          60
1          60
2          60
3          60
4          60


In [38]:
categorical_maps

{'department': {'Agricultural Engineering': 1,
  'Agricultural Sciences': 2,
  'Agricultural and Biological Engineering': 3,
  'Agricultural and Environmental Sciences': 4,
  'Agronomy': 5,
  'Amity Institute of Applied Sciences (Noida)': 6,
  'Amity School of Physical Sciences': 7,
  'Amrut Mody School of Management': 8,
  'Biochemistry': 9,
  'Bioinformatics': 10,
  'Bioinformatics and Computational Biology': 11,
  'Biological Engineering': 12,
  'Biological Sciences': 13,
  'Chemical Engineering': 14,
  'Chemical and Biochemical Processing Division': 15,
  'Chemistry': 16,
  'Civil and Environmental Engineering': 17,
  'Computational Biology': 18,
  'Computer Science': 19,
  'Computer Science and Artificial Intelligence Laboratory': 20,
  'Computer Science and Artificial Intelligence Laboratory (CSAIL)': 21,
  'Computer Science and Engineering': 22,
  'Computer Science and Media Arts': 23,
  'Computer Science and Media Lab': 24,
  'Computer Science and Technology': 25,
  'Department

In [ ]:

# # Categorical data preprocessing
# categorical_maps = {}

# for col in categorical_columns:
#     values = scholar_df[col].fillna("UNKNOWN").astype(str)
#     unique_values = sorted(values.unique())
#     # mapping  = {value: idx for idx, value in enumerate(unique_values)}
#     mapping = {value: idx for idx, value in enumerate(unique_values) }

#     # Reserve 0 for UNKNOWN
#     mapping = {value: idx + 1 for idx, value in enumerate(unique_values)}
#     mapping["UNKNOWN"] = 0
#     categorical_maps[col] = mapping

In [40]:

import os
os.makedirs("src/data/processed/vocab_dict_data", exist_ok=True)
with open("src/data/processed/vocab_dict_data/categorical_maps.json", "w") as file:
    json.dump(categorical_maps, file, indent=4)

In [41]:
import os
import json

# Get current directory of the notebook kernel
cwd = os.getcwd()

# If running inside 'src/utils', go up two levels to the project root
if cwd.endswith("utils") or "src\\utils" in cwd:
    project_root = os.path.abspath(os.path.join(cwd, "..", ".."))
else:
    project_root = cwd

# Construct absolute path to the target directory
save_dir = os.path.join(project_root, "src", "data", "processed", "vocab_dict_data")
os.makedirs(save_dir, exist_ok=True)

# Save the JSON file
file_path = os.path.join(save_dir, "categorical_maps.json")
with open(file_path, "w") as file:
    json.dump(categorical_maps, file, indent=4)

print(f"Saved successfully to: {file_path}")


Saved successfully to: c:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\vocab_dict_data\categorical_maps.json
